### Source Tables:
- `_exponent._bronze_epic_clarity.patient` — patient demographics including address

### Strategy:
- Address fields are directly on the patient table (ADD_LINE_1, ADD_LINE_2, CITY, STATE_C, ZIP, COUNTY_C, COUNTRY_C)
- STATE_C is Epic internal code — mapped via CASE to state abbreviations
- COUNTY_C is Epic internal code — stored as-is (no zc_county lookup available)
- COUNTRY_C is Epic internal code — stored as-is (no zc_country lookup available)
- One address per patient (no deduplication needed — address is on the patient row)
- Filter: ADD_LINE_1 IS NOT NULL

### Notes:
- Epic state code mapping derived from sample city analysis
- ~10M patient addresses expected

In [0]:
%sql
-- Create silver temp view for Epic Clarity location
CREATE OR REPLACE TEMPORARY VIEW silver AS
SELECT
  LOWER(TRIM(p.ADD_LINE_1)) AS address_1,
  LOWER(TRIM(p.ADD_LINE_2)) AS address_2,
  LOWER(TRIM(p.CITY)) AS city,
  LOWER(
    CASE CAST(p.STATE_C AS INT)
      WHEN 1 THEN 'AL' WHEN 2 THEN 'AK' WHEN 3 THEN 'AZ' WHEN 4 THEN 'AR'
      WHEN 5 THEN 'CA' WHEN 6 THEN 'CO' WHEN 7 THEN 'CT' WHEN 8 THEN 'DE'
      WHEN 9 THEN 'DC' WHEN 10 THEN 'FL' WHEN 11 THEN 'GA' WHEN 12 THEN 'HI'
      WHEN 13 THEN 'ID' WHEN 14 THEN 'IL' WHEN 15 THEN 'IN' WHEN 16 THEN 'IA'
      WHEN 17 THEN 'KS' WHEN 18 THEN 'KY' WHEN 19 THEN 'LA' WHEN 20 THEN 'ME'
      WHEN 21 THEN 'MD' WHEN 22 THEN 'MA' WHEN 23 THEN 'MI' WHEN 24 THEN 'MN'
      WHEN 25 THEN 'MS' WHEN 26 THEN 'MO' WHEN 27 THEN 'MT' WHEN 28 THEN 'NE'
      WHEN 29 THEN 'NV' WHEN 30 THEN 'NH' WHEN 31 THEN 'NJ' WHEN 32 THEN 'NM'
      WHEN 33 THEN 'NY' WHEN 34 THEN 'NC' WHEN 35 THEN 'ND' WHEN 36 THEN 'OH'
      WHEN 37 THEN 'OK' WHEN 38 THEN 'OR' WHEN 39 THEN 'PA' WHEN 40 THEN 'RI'
      WHEN 41 THEN 'SC' WHEN 42 THEN 'SD' WHEN 43 THEN 'TN' WHEN 44 THEN 'TX'
      WHEN 45 THEN 'UT' WHEN 46 THEN 'VT' WHEN 47 THEN 'VA' WHEN 48 THEN 'WA'
      WHEN 49 THEN 'WV' WHEN 50 THEN 'WI' WHEN 51 THEN 'WY'
      WHEN 52 THEN 'PR' WHEN 53 THEN 'VI' WHEN 54 THEN 'GU' WHEN 55 THEN 'AS'
      ELSE CAST(p.STATE_C AS STRING)
    END
  ) AS state,
  LOWER(TRIM(p.ZIP)) AS zip,
  CAST(p.COUNTY_C AS STRING) AS county,
  CONCAT_WS(CHR(31), 'epic_clarity', 'patient', 'PAT_ID', p.PAT_ID) AS location_source_value,
  0 AS country_concept_id,
  CAST(p.COUNTRY_C AS STRING) AS country_source_value,
  NULL AS latitude,
  NULL AS longitude,
  'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.patient p
WHERE p.ADD_LINE_1 IS NOT NULL

In [0]:
# %sql
# -- Preview silver
# SELECT * FROM silver LIMIT 10
## -- note country_source_value 1 or null

In [0]:
%sql
MERGE INTO _exponent.omop_silver.location AS target
USING silver AS source
ON target.location_source_value = source.location_source_value

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.country_source_value   <=> source.country_source_value
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.source_system          <=> source.source_system
) THEN UPDATE SET
  target.address_1              = source.address_1,
  target.address_2              = source.address_2,
  target.city                   = source.city,
  target.state                  = source.state,
  target.zip                    = source.zip,
  target.county                 = source.county,
  target.country_concept_id     = source.country_concept_id,
  target.country_source_value   = source.country_source_value,
  target.latitude               = source.latitude,
  target.longitude              = source.longitude,
  target.source_system          = source.source_system,
  target.last_mod_tsp           = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  country_source_value,
  latitude,
  longitude,
  source_system,
  last_mod_tsp
) VALUES (
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.country_source_value,
  source.latitude,
  source.longitude,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_location (
    source_system,
    location_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_location.source_system,
    silver_location.location_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(silver_location.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        location_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.location
    WHERE source_system = 'epic_clarity'
      AND location_source_value IS NOT NULL
) AS silver_location
LEFT ANTI JOIN _exponent.omop_mapping.source_to_location AS existing_location
  ON silver_location.location_source_value = existing_location.location_source_value;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_location.location_id,
  location.address_1,
  location.address_2,
  location.city,
  location.state,
  location.zip,
  location.county,
  location.country_concept_id,
  location.latitude,
  location.longitude,
  location.location_source_value,
  location.last_mod_tsp
FROM _exponent.omop_silver.location AS location
JOIN _exponent.omop_mapping.source_to_location AS source_to_location
  ON location.location_source_value = source_to_location.location_source_value
 AND source_to_location.active_flag = TRUE
WHERE location.source_system = 'epic_clarity'

In [0]:
%sql
MERGE INTO _exponent.omop.location AS target
USING gold AS source
ON target.location_id = source.location_id

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.location_source_value  <=> source.location_source_value
) THEN UPDATE SET
  target.address_1               = source.address_1,
  target.address_2               = source.address_2,
  target.city                    = source.city,
  target.state                   = source.state,
  target.zip                     = source.zip,
  target.county                  = source.county,
  target.country_concept_id      = source.country_concept_id,
  target.latitude                = source.latitude,
  target.longitude               = source.longitude,
  target.location_source_value   = source.location_source_value

WHEN NOT MATCHED THEN INSERT (
  location_id,
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  latitude,
  longitude
) VALUES (
  source.location_id,
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.latitude,
  source.longitude
);

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.location WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_location WHERE source_system = 'epic_clarity'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.location WHERE location_source_value LIKE 'epic_clarity%'

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.location
# WHERE location_source_value LIKE 'epic_clarity%'
# LIMIT 10